# Qwen2.5-VL Step-DPO Fine-Tuning (Kaggle GPU)

This notebook trains a QLoRA adapter on the extracted Step-DPO pairs using TRL's `DPOTrainer` to align the model's step-by-step reasoning logic.

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    os.environ['HF_TOKEN'] = user_secrets.get_secret('HF_TOKEN')
    print("Successfully loaded HF_TOKEN secrets from Kaggle.")
except Exception as e:
    print("Could not load Kaggle secrets. Skipping token setup.")

if not os.path.exists("prm_project"):
    !git clone https://github.com/yahorlahunovich/prm_project.git
    %cd prm_project
else:
    %cd prm_project
    !git pull

In [ ]:
# Install packages with --no-deps to preserve pre-installed Kaggle PyTorch (maintains P100 sm_60 and T4 sm_75 CUDA support)
!pip install -q --no-deps "transformers>=4.49.0" "trl>=0.12.0" qwen-vl-utils

In [ ]:
import sys
import traceback
import json
import os
import torch
from datasets import Dataset
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
from peft import LoraConfig
from trl import DPOTrainer, DPOConfig
from PIL import Image

try:
    # 1. Check & Auto-Download Images if missing
    images_dir = "data/CharXiv/images"
    if not os.path.exists(images_dir) or len(os.listdir(images_dir)) == 0:
        print("Images directory empty or missing. Auto-downloading required chart images...")
        os.system("python scripts/download_images.py")

    # 2. Load Dataset
    data_path = "experiments/001_500_reasoning/data/step_dpo_pairs.jsonl"
    with open(data_path, 'r') as f:
        raw_data = [json.loads(line) for line in f]
        
    hf_data = {"prompt": [], "chosen": [], "rejected": [], "images": []}
    valid_count = 0

    for item in raw_data:
        abs_path = os.path.abspath(item["image_path"])
        if not os.path.exists(abs_path):
            print(f"Warning: Image not found at {abs_path}. Auto-downloading images...")
            os.system("python scripts/download_images.py")
        
        try:
            img = Image.open(abs_path).convert("RGB")
        except Exception as e:
            print(f"Error opening image {abs_path}: {e}. Skipping sample.")
            continue
            
        question_text = item.get("question", "").strip()
        prompt_text = "Analyze this chart. Provide step-by-step reasoning and a final answer.\n" + question_text
        
        prompt_msg = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": prompt_text}
                ]
            }
        ]
        
        prefix = item.get("prefix", "")
        full_chosen = prefix + item["chosen"] + "\n"
        full_rejected = prefix + item["rejected"] + "\n"
        
        chosen_msg = [{"role": "assistant", "content": full_chosen}]
        rejected_msg = [{"role": "assistant", "content": full_rejected}]
        
        hf_data["prompt"].append(prompt_msg)
        hf_data["chosen"].append(chosen_msg)
        hf_data["rejected"].append(rejected_msg)
        hf_data["images"].append([img])
        valid_count += 1

    dataset = Dataset.from_dict(hf_data)
    print(f"Successfully loaded {valid_count} valid Step-DPO pairs.")

    # 3. Load Model & Processor in FP16
    model_id = "Qwen/Qwen2.5-VL-3B-Instruct"

    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        attn_implementation="sdpa",
        device_map={"": 0}
    )

    model.enable_input_require_grads()

    # Freeze Vision Tower
    if hasattr(model, "visual"):
        model.visual.requires_grad_(False)

    processor = AutoProcessor.from_pretrained(model_id, min_pixels=256*28*28, max_pixels=512*28*28)
    processor.tokenizer.padding_side = 'right'
    if processor.tokenizer.pad_token is None:
        processor.tokenizer.pad_token = processor.tokenizer.eos_token
        
    # Expose pad/eos tokens on processor object for DPOTrainer compatibility
    processor.pad_token = processor.tokenizer.pad_token
    processor.pad_token_id = processor.tokenizer.pad_token_id
    processor.eos_token_id = processor.tokenizer.eos_token_id

    # 4. Configure LoRA
    peft_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        task_type="CAUSAL_LM",
    )

    # 5. Define DPO Trainer
    training_args = DPOConfig(
        output_dir="./dpo_qwen_vl",
        beta=0.1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        learning_rate=1e-5,
        num_train_epochs=3,
        max_length=2048,
        logging_steps=5,
        save_steps=25,
        save_total_limit=2,
        gradient_checkpointing=True,
        dataset_num_proc=1,
        remove_unused_columns=False,
        report_to="none",
        fp16=True,
        bf16=False
    )

    print("\n=== INITIALIZING DPOTRAINER ===")
    trainer = DPOTrainer(
        model,
        ref_model=None,
        args=training_args,
        train_dataset=dataset,
        processing_class=processor,
        peft_config=peft_config,
    )
    print("\n=== DPOTRAINER INITIALIZATION COMPLETE ===")

    # 6. Train and Save
    print("\n=== STARTING TRAINING LOOP ===")
    trainer.train()
    trainer.save_model("qwen_vl_step_dpo_adapter")
    print("Training complete and adapter saved.")

except Exception as e:
    err_trace = traceback.format_exc()
    print("=== FATAL EXECUTION ERROR ===")
    print(err_trace)
    with open("execution_error.log", "w") as f_err:
        f_err.write(err_trace)
    raise e